# Part A2 - Segmenting the Drop-off, Finding the Biggest Fixable Leak

**What this notebook does:** builds on A1's result to answer **A2** - which cuts (phone type, sign-up channel, city, vehicle type) actually explain the drop-off, and by how much. For every real leak found, we size it as "how many more approved captains per month if we fixed this" — exactly what the brief asks for.

### Step 0 - Load the same rulebook as A1
Uses the exact same loading and rules as A1, so the two notebooks can never quietly disagree with each other.

In [1]:
import pandas as pd
import common

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_DIR = common.find_data_dir()
captains, doc_events, approvals, activation, nudges = common.load_all(DATA_DIR)
master, master_mature, master_recent = common.build_master(captains, approvals, activation)
print(f"master_mature: {len(master_mature):,} rows (same denominator as A1)")

master_mature: 20,207 rows (same denominator as A1)


### Step 1 - Bring in each captain's outcome and reason for leaving
Reuses A1's result so we can now split it by city, phone type, sign-up channel, and vehicle type - the cuts the brief tells us to consider for **A2, the biggest fixable leak**.

In [2]:
doc_events_mature = doc_events[doc_events["captain_id"].isin(master_mature["captain_id"])]
result = common.classify_doc_loss_reason(master_mature, doc_events_mature)
result["approved_flag"] = result["loss_bucket"] == "approved"
print(result["loss_bucket"].value_counts())

loss_bucket
failed_gave_up        8887
stalled_no_failure    5967
approved              3532
never_started         1248
post_doc_rejection     356
exhausted_attempts     217
Name: count, dtype: int64


### Step 2 - Try every likely cut
The brief asks us to **segment the drop-off** and decide which cuts are worth making. This checks all four candidates side by side, before deciding which ones actually matter.

In [3]:
segment_tables = {}
for col in ["device_tier", "acquisition_channel", "vehicle_type", "city"]:
    g = result.groupby(col).agg(n=("captain_id", "size"), approved=("approved_flag", "sum"))
    g["approval_rate"] = g["approved"] / g["n"]
    g["share_of_signups"] = g["n"] / g["n"].sum()
    segment_tables[col] = g
    print(f"\n=== Approval rate by {col} ===")
    print(g.sort_values("approval_rate").to_string())


=== Approval rate by device_tier ===
                n  approved  approval_rate  share_of_signups
device_tier                                                 
low          9290      1296       0.139505          0.459742
mid          7744      1510       0.194990          0.383234
high         3173       726       0.228806          0.157025

=== Approval rate by acquisition_channel ===
                        n  approved  approval_rate  share_of_signups
acquisition_channel                                                 
paid_digital         2830       283       0.100000          0.140050
gc_telecalling       1176       172       0.146259          0.058198
organic_app          6758      1015       0.150192          0.334439
referral             4431       722       0.162943          0.219280
fos_field            5012      1340       0.267358          0.248033

=== Approval rate by vehicle_type ===
                 n  approved  approval_rate  share_of_signups
vehicle_type               

### Stated assumption - how we turn a percentage into 'per month'
The brief says undeclared assumptions are not OK. Sign-ups have been growing, so we state clearly which sign-up volume (this month's, not an old average) we use to size every monthly number below.

In [4]:
HISTORICAL_AVG, CURRENT_RUNRATE = common.monthly_signup_volume(captains)
print(f"Historical average (Jan-May): {HISTORICAL_AVG:,.0f} signups/month")
print(f"Current run-rate (June, most recent complete month): {CURRENT_RUNRATE:,.0f} signups/month")
print(f"Growth over the window: {(CURRENT_RUNRATE/captains.groupby('signup_month').size().iloc[0] - 1):.1%}")
print(f"\nUsing CURRENT_RUNRATE as the primary basis for every projection below --")
print(f"using the historical average instead would understate each figure by")
print(f"{(1 - HISTORICAL_AVG/CURRENT_RUNRATE):.1%}.")

Historical average (Jan-May): 4,041 signups/month
Current run-rate (June, most recent complete month): 4,793 signups/month
Growth over the window: 29.0%

Using CURRENT_RUNRATE as the primary basis for every projection below --
using the historical average instead would understate each figure by
15.7%.


### Biggest fixable leak, part 1 - phone quality
Sizes the gap in plain terms: if cheaper phones approved captains at the same rate as good phones, how many more approved captains would we get every month?

In [5]:
by_device = segment_tables["device_tier"]
low_share = by_device.loc["low", "n"] / by_device["n"].sum()
low_monthly_current = CURRENT_RUNRATE * low_share
low_monthly_historical = HISTORICAL_AVG * low_share
uplift_device = by_device.loc["high", "approval_rate"] - by_device.loc["low", "approval_rate"]

extra_approvals_current = low_monthly_current * uplift_device
extra_approvals_historical = low_monthly_historical * uplift_device

print(f"low-tier share of signups: {low_share:.1%}")
print(f"high-tier share of signups: {by_device.loc['high','n']/by_device['n'].sum():.1%}")
print(f"Approval-rate gap (high - low): {uplift_device:.1%}")
print()
print(f"low-tier monthly signups (current run-rate basis): {low_monthly_current:.0f}")
print(f"IF low-tier matched high-tier's rate -> extra approvals/month: {extra_approvals_current:.0f}   <- PRIMARY")
print()
print(f"[Comparison, historical-average basis: {extra_approvals_historical:.0f} -- "
      f"{(1-extra_approvals_historical/extra_approvals_current):.0%} lower, understates current opportunity]")
print()
print("(Assumes device tier is causal, not just correlated with something else that")
print("drives approval -- flagged in the memo's assumptions section.)")

low-tier share of signups: 46.0%
high-tier share of signups: 15.7%
Approval-rate gap (high - low): 8.9%

low-tier monthly signups (current run-rate basis): 2204
IF low-tier matched high-tier's rate -> extra approvals/month: 197   <- PRIMARY

[Comparison, historical-average basis: 166 -- 16% lower, understates current opportunity]

(Assumes device tier is causal, not just correlated with something else that
drives approval -- flagged in the memo's assumptions section.)


### Biggest fixable leak, part 2 - how people signed up
Same idea, a different cut: sizes the gap between sign-up channels, and checks it isn't just the phone-quality story showing up twice (a **confound check**).

In [6]:
by_channel = segment_tables["acquisition_channel"]
paid_digital_share = by_channel.loc["paid_digital", "n"] / by_channel["n"].sum()
paid_digital_monthly_current = CURRENT_RUNRATE * paid_digital_share
uplift_channel = by_channel.loc["fos_field", "approval_rate"] - by_channel.loc["paid_digital", "approval_rate"]
extra_approvals_channel = paid_digital_monthly_current * uplift_channel

print(f"paid_digital share of signups: {paid_digital_share:.1%}")
print(f"paid_digital monthly signups (current run-rate basis): {paid_digital_monthly_current:.0f}")
print(f"Approval-rate gap (fos_field - paid_digital): {uplift_channel:.1%}")
print(f"IF paid_digital matched fos_field's rate -> extra approvals/month: {extra_approvals_channel:.0f}   <- PRIMARY")

ct = pd.crosstab(master_mature["acquisition_channel"], master_mature["device_tier"], normalize="index")
print("\nDevice-tier mix within each channel (row %) -- near-identical confirms independence:")
print(ct.round(3).to_string())

paid_digital share of signups: 14.0%
paid_digital monthly signups (current run-rate basis): 671
Approval-rate gap (fos_field - paid_digital): 16.7%
IF paid_digital matched fos_field's rate -> extra approvals/month: 112   <- PRIMARY

Device-tier mix within each channel (row %) -- near-identical confirms independence:
device_tier           high    low    mid
acquisition_channel                     
fos_field            0.158  0.463  0.379
gc_telecalling       0.144  0.456  0.401
organic_app          0.158  0.468  0.374
paid_digital         0.154  0.454  0.392
referral             0.160  0.449  0.391


### Checked, but ruled out
The brief asks what we chose not to chase, and why. This checks city and vehicle type and shows neither explains much - so we're not spending effort there.

In [7]:
g = segment_tables["city"]
lo, hi = g["approval_rate"].min(), g["approval_rate"].max()
print(g.sort_values("approval_rate").to_string())
print(f"\nRange: {lo:.1%} to {hi:.1%}  |  Spread: {(hi-lo)*100:.1f} points -- too narrow to act on")
print()
print("Vehicle type: ERickshaw's higher approval rate is explained structurally")
print("(5 required docs vs. 6 -- no Permit gate), not a fixable process issue.")

              n  approved  approval_rate  share_of_signups
city                                                      
Pune       4379       727       0.166020          0.216707
Hyderabad  6193      1040       0.167932          0.306478
Bangalore  4420       755       0.170814          0.218736
Delhi      5215      1010       0.193672          0.258079

Range: 16.6% to 19.4%  |  Spread: 2.8 points -- too narrow to act on

Vehicle type: ERickshaw's higher approval rate is explained structurally
(5 required docs vs. 6 -- no Permit gate), not a fixable process issue.


### Summary for A2

| Cut | Gap | Extra approved captains/month if fixed |
|---|---|---|
| Phone quality (cheap vs good) | 8.9 points | **~197** |
| Sign-up channel (paid ad vs field agent) | 16.7 points | **~112** |

Both are checked to be separate problems, not the same one counted twice — so they need two separate fixes, worth roughly **~309 more approved captains/month** combined.